In [1]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix
from IPython.display import display
from finder_core.public_outputs import sequential_thresholds
n_bootstrap = 10000
block_length = 21
confidence_level = 0.95
random_seed = 42
output = Path('Results/Confidence Intervals')
scores = pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')


In [2]:
markets = {'^GSPC': 'SP500', 'FTSE100': 'FTSE', 'GDAXI': 'DAX', 'CACT': 'CAC', 'STOXX': 'STOXX', 'CBU0.L': 'CBU0'}
models = ['decision_supervisor', 'quantitative_expert', 'chart_expert', 'news_reader', 'news_expert', 'lstm_ind', 'lstm_news', 'logistic_ind', 'logistic_news', 'gjr_garch']
metric_names = ['roc_auc', 'ap']
frames, metric_rows, decision_rows = [], [], []
gjr = pd.read_csv('Benchmark/Official/gjr_predictions.csv')
point_metrics = np.empty((len(markets), len(models), len(metric_names)))
for m, (market, code) in enumerate(markets.items()):
    frame = scores.loc[scores.market.eq(market), ['target_date', 'information_cutoff', 'actual_label', *models[:5]]].copy()
    for variant, suffix in [('IND', 'ind'), ('NEWS', 'news')]:
        benchmark = pd.read_csv(Path('Benchmark/Official') / f'{code}_{variant}' / 'test_predictions.csv')
        frame = frame.merge(benchmark[['target_date', 'actual', 'probability', 'logistic_probability']], on='target_date', validate='one_to_one')
        assert frame.actual_label.eq(frame.actual).all(), (market, variant)
        frame = frame.drop(columns='actual').rename(columns={'probability': 'lstm_' + suffix, 'logistic_probability': 'logistic_' + suffix})
    frame = frame.merge(gjr.loc[gjr.market.eq(market), ['target_date', 'actual', 'garch_probability']], on='target_date', validate='one_to_one')
    assert frame.actual_label.eq(frame.actual).all(), (market, 'GJR')
    frame = frame.drop(columns='actual').rename(columns={'garch_probability': 'gjr_garch'}).sort_values('target_date').reset_index(drop=True)
    assert not frame.target_date.duplicated().any() and np.isfinite(frame[models]).all().all()
    y = frame.actual_label.to_numpy()
    for j, model in enumerate(models):
        probability = frame[model].to_numpy()
        thresholds = sequential_thresholds(probability, y, frame.target_date, frame.information_cutoff)
        predicted = (probability >= thresholds).astype(int)
        tn, fp, fn, tp = map(int, confusion_matrix(y, predicted, labels=[0, 1]).ravel())
        point_metrics[m, j] = [roc_auc_score(y, probability), average_precision_score(y, probability)]
        metric_rows.append({'market': market, 'model': model, 'n': len(y), 'events': int(y.sum()), 'roc_auc': point_metrics[m, j, 0], 'ap': point_metrics[m, j, 1], 'accuracy': accuracy_score(y, predicted), 'precision': precision_score(y, predicted, zero_division=0), 'recall': recall_score(y, predicted, zero_division=0), 'f1': f1_score(y, predicted, zero_division=0), 'mcc': matthews_corrcoef(y, predicted), 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn, 'initial_threshold': 0.5, 'last_applied_threshold': thresholds[-1], 'min_threshold': thresholds.min(), 'max_threshold': thresholds.max(), 'threshold_method': 'expanding_past_resolved_f1'})
        history = []
        for i in range(len(frame)):
            eligible = (np.arange(len(frame)) < i) & (frame.target_date.to_numpy() <= frame.information_cutoff.iloc[i])
            previous_threshold = thresholds[i - 1] if i else 0.5
            candidates = np.unique(np.r_[0., 1., 0.5, previous_threshold, probability[eligible]]) if y[eligible].sum() > 0 else np.array([previous_threshold])
            history.append({'history_n': int(eligible.sum()), 'history_events': int(y[eligible].sum()), 'history_last_target': frame.loc[eligible, 'target_date'].max() if eligible.any() else '', 'historical_f1': f1_score(y[eligible], probability[eligible] >= thresholds[i], zero_division=0) if eligible.any() else 0., 'threshold_candidates': len(candidates)})
        decision = frame[['target_date', 'information_cutoff']].assign(market=market, agent=model, score=probability, actual_label=y, threshold=thresholds, prediction=predicted)
        decision_rows.append(pd.concat([decision, pd.DataFrame(history)], axis=1))
    frames.append(frame)
calculated = pd.DataFrame(metric_rows)
decisions = pd.concat(decision_rows, ignore_index=True)
display(calculated)


     market                model  ...  max_threshold            threshold_method
0     ^GSPC  decision_supervisor  ...       0.712300  expanding_past_resolved_f1
1     ^GSPC  quantitative_expert  ...       0.720000  expanding_past_resolved_f1
2     ^GSPC         chart_expert  ...       0.680000  expanding_past_resolved_f1
3     ^GSPC          news_reader  ...       0.712300  expanding_past_resolved_f1
4     ^GSPC          news_expert  ...       0.735000  expanding_past_resolved_f1
5     ^GSPC             lstm_ind  ...       0.621753  expanding_past_resolved_f1
6     ^GSPC            lstm_news  ...       0.641317  expanding_past_resolved_f1
7     ^GSPC         logistic_ind  ...       0.500000  expanding_past_resolved_f1
8     ^GSPC        logistic_news  ...       0.500000  expanding_past_resolved_f1
9     ^GSPC            gjr_garch  ...       0.500000  expanding_past_resolved_f1
10  FTSE100  decision_supervisor  ...       0.785000  expanding_past_resolved_f1
11  FTSE100  quantitative_ex

In [3]:
bootstrap_metrics = np.empty((n_bootstrap, len(markets), len(models), len(metric_names)))
for m, frame in enumerate(frames):
    rng = np.random.default_rng(random_seed)
    y = frame.actual_label.to_numpy()
    probabilities = frame[models].to_numpy()
    n = len(frame)
    assert 1 <= block_length <= n and np.unique(y).size == 2
    starts = np.arange(n - block_length + 1)
    valid, attempts = 0, 0
    while valid < n_bootstrap and attempts < n_bootstrap * 20:
        attempts += 1
        selected_starts = rng.choice(starts, size=int(np.ceil(n / block_length)), replace=True)
        indices = np.concatenate([np.arange(start, start + block_length) for start in selected_starts])[:n]
        actual = y[indices]
        if np.unique(actual).size < 2:
            continue
        for j in range(len(models)):
            probability = probabilities[indices, j]
            bootstrap_metrics[valid, m, j] = [roc_auc_score(actual, probability), average_precision_score(actual, probability)]
        valid += 1
    assert valid == n_bootstrap, (list(markets)[m], valid, attempts)
    print(list(markets)[m], valid, attempts)
point_differences = point_metrics[:, :1] - point_metrics[:, 1:]
bootstrap_differences = bootstrap_metrics[:, :, :1] - bootstrap_metrics[:, :, 1:]


^GSPC 10000 10000
FTSE100 10000 10000
GDAXI 10000 10000
CACT 10000 10000
STOXX 10000 10000
CBU0.L 10000 10000


In [4]:
interval_rows, difference_rows = [], []
quantiles = [(1 - confidence_level) / 2, (1 + confidence_level) / 2]
settings = {'method': 'moving_block_bootstrap_percentile', 'confidence_level': confidence_level, 'n_bootstrap': n_bootstrap, 'block_length': block_length, 'block_unit': 'market_observations', 'random_seed': random_seed}
for m, market in enumerate(markets):
    context = {'market': market, 'n': len(frames[m]), 'events': int(frames[m].actual_label.sum()), 'event_rate': frames[m].actual_label.mean(), 'aggregation': 'within_market'}
    for j, model in enumerate(models):
        for k, metric in enumerate(metric_names):
            bounds = np.quantile(bootstrap_metrics[:, m, j, k], quantiles)
            interval_rows.append({**context, 'model': model, 'metric': metric, 'estimate': point_metrics[m, j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], 'n_valid': n_bootstrap, 'n_invalid': 0, **settings})
    for j, comparator in enumerate(models[1:]):
        for k, metric in enumerate(metric_names):
            bounds = np.quantile(bootstrap_differences[:, m, j, k], quantiles)
            difference_rows.append({**context, 'model': models[0], 'comparator': comparator, 'metric': metric, 'estimate': point_differences[m, j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], 'n_valid': n_bootstrap, 'n_invalid': 0, **settings})
metrics_ci = pd.DataFrame(interval_rows)
paired_differences_ci = pd.DataFrame(difference_rows)
output.mkdir(parents=True, exist_ok=True)
for folder in [output, Path('FINDER/Uncertainty')]:
    metrics_ci.to_csv(folder / 'metrics_ci.csv', index=False)
    paired_differences_ci.to_csv(folder / 'paired_differences_ci.csv', index=False)
calculated.loc[calculated.model.isin(models[:5])].rename(columns={'model': 'agent'}).to_csv('FINDER/Official/market_agent_metrics.csv', index=False)
decisions.loc[decisions.agent.isin(models[:5])].to_csv('FINDER/Official/daily_thresholds.csv', index=False)
benchmark_metrics = calculated.loc[calculated.model.isin(models[5:])].copy()
benchmark_metrics['market'] = benchmark_metrics.market.map(markets)
benchmark_metrics['model'] = benchmark_metrics.model.map({'lstm_ind': 'IND', 'lstm_news': 'NEWS', 'logistic_ind': 'Logistic_IND', 'logistic_news': 'Logistic_NEWS', 'gjr_garch': 'GJR_GARCH'})
benchmark_metrics.to_csv('Benchmark/Official/metrics.csv', index=False)
display(metrics_ci.loc[metrics_ci.model.eq('decision_supervisor'), ['market', 'metric', 'estimate', 'ci_lower', 'ci_upper']].round(4))


      market   metric  estimate  ci_lower  ci_upper
0      ^GSPC  roc_auc    0.7107    0.4628    0.8358
1      ^GSPC       ap    0.1371    0.0275    0.2335
20   FTSE100  roc_auc    0.6763    0.5408    0.7918
21   FTSE100       ap    0.2482    0.0604    0.3912
40     GDAXI  roc_auc    0.5924    0.3868    0.7245
41     GDAXI       ap    0.1184    0.0378    0.2199
60      CACT  roc_auc    0.6754    0.5495    0.8074
61      CACT       ap    0.1687    0.0505    0.3033
80     STOXX  roc_auc    0.6443    0.5268    0.7807
81     STOXX       ap    0.1358    0.0444    0.2498
100   CBU0.L  roc_auc    0.6581    0.4231    0.7903
101   CBU0.L       ap    0.0839    0.0366    0.1629


In [5]:
evaluate_new_run = False
run_id = 'concept'
ticker = '^GSPC'

if evaluate_new_run:
    from finder_core.settings import load_settings
    from finder_core.public_outputs import evaluate
    settings = load_settings(root, ticker=ticker)
    scores, metrics = evaluate(root / 'Runs' / ticker / run_id, root / settings['paths']['labels'])
    display(metrics)
